**1. 감정 강도 기반 감정 단어 추출 (감정 사전용 핵심 분석)**

📌 목적: 감정세기 높은 문장에서 자주 등장하는 단어를 추출 → 감정사전 후보

🎯 어떻게?

감정세기 ≥ 2인 샘플 필터링

각 감정별로 상위 빈도 단어 추출 (angry, happiness, sad, surprise 등)

예: angry + 세기 2 이상인 문장에서 "짜증", "빡쳐", "최악", "개같", "환장" 같은 단어 뽑기

In [ ]:
import pandas as pd
import re
from collections import Counter, defaultdict
from tqdm import tqdm

# 1. 파일 불러오기
df = pd.read_csv("/content/5차년도_2차.csv", encoding='cp949')

# 2. 분석 대상 컬럼
text_col = "발화문"
emotion_cols = ['1번 감정', '1번 감정세기',
                '2번 감정', '2번 감정세기',
                '3번 감정', '3번 감정세기',
                '4번 감정', '4번감정세기',
                '5번 감정', '5번 감정세기']

# 3. 감정별 단어 카운터 만들기
emotion_word_counters = defaultdict(Counter)

# 4. 정규식 전처리 함수
def clean_text(text):
    return re.sub(r"[^가-힣\s]", "", str(text))

# 5. 각 문장마다 감정세기 2 이상인 감정들에 대해 단어 추출
for _, row in tqdm(df.iterrows(), total=len(df)):
    cleaned = clean_text(row[text_col])
    words = cleaned.split()

    for i in range(0, len(emotion_cols), 2):
        emotion = row[emotion_cols[i]]
        strength = row[emotion_cols[i+1]]

        if pd.notna(emotion) and pd.notna(strength) and int(strength) >= 2:
            emotion_word_counters[emotion].update([w for w in words if len(w) > 1])

# 6. 감정별 상위 단어 출력
for emotion, counter in emotion_word_counters.items():
    print(f"\n💥 감정: {emotion}")
    for word, count in counter.most_common(20):
        print(f"{word}: {count}")


100%|██████████| 19374/19374 [00:03<00:00, 5783.30it/s] 


💥 감정: angry
아니: 439
짜증나: 436
너무: 356
집에나: 345
곰팡이가: 340
나한테만: 326
새끼는: 319
나한테: 298
청소를: 290
거야: 259
것이지: 257
그래: 249
화가: 240
요즘: 240
해도: 236
않아: 228
화장실은: 223
정말: 204
뭐라고: 202
자꾸: 190

💥 감정: surprise
깜짝: 708
놀랐어: 517
아까: 421
아니: 342
해피: 323
갑자기: 322
정말: 294
산책시키다가: 283
자다가: 271
어제: 261
목줄이: 244
놀랬어: 231
엄청: 218
해피를: 177
해피가: 156
놀라서: 152
이벤트: 125
당첨됐어: 124
깼어: 121
거야: 115

💥 감정: happiness
내가: 1003
너무: 669
끝났어: 600
드디어: 595
좋은: 500
기분: 457
마라톤: 411
축하해줘: 401
이벤트에: 358
일이: 357
프로젝트: 354
완전히: 320
있어: 314
당첨: 307
정말: 275
달리는: 270
거야: 264
마음에: 256
엄청: 241
인플루언서가: 239

💥 감정: sadness
너무: 1462
우울해: 786
많이: 385
때문에: 370
없어: 348
같아: 345
코로나: 333
있어: 328
계속: 323
요즘: 299
비가: 287
내가: 257
맞아: 253
밖에: 238
지금: 227
정말: 220
몸이: 217
속상해: 203
있으니까: 182
곰팡이가: 178

💥 감정: disgust
냄새가: 861
역겨운: 560
쓰레기통에서: 554
있어: 395
나고: 368
너무: 305
내가: 249
짜장면에서: 218
벌레가: 212
곰팡이가: 196
청소를: 194
해도: 167
음식물: 154
시작했어: 153
나기: 152
않아: 150
나왔어: 150
화장실은: 144
아니: 137
같아: 131

💥 감정: fear
같아: 473
혼자: 376
갇히게: 371
너무: 347
무서워: 3

**2. 감정별 대표 접미어 / 접두어 패턴 분석 (서브워드화 힌트)**

📌 목적: 감정 표현에 자주 쓰이는 공통적인 서브워드 단위 분석

🎯 어떻게?

감정이 강한 문장들에서 3~5글자 단어를 분리하여 접두/접미 패턴 분석

예: "슬픔", "슬프다", "슬퍼서", "슬펐" → 슬프, -다, -서 같은 파생형 확인

In [ ]:
import pandas as pd
import re
from collections import Counter, defaultdict
from tqdm import tqdm

# 1. 데이터 불러오기
df = pd.read_csv("/content/5차년도_2차.csv", encoding='cp949')

# 2. 감정 컬럼 정의
text_col = "발화문"
emotion_cols = ['1번 감정', '1번 감정세기',
                '2번 감정', '2번 감정세기',
                '3번 감정', '3번 감정세기',
                '4번 감정', '4번감정세기',
                '5번 감정', '5번 감정세기']

# 3. 한글만 남기기 위한 전처리 함수
def clean_text(text):
    return re.sub(r"[^가-힣\s]", "", str(text))

# 4. 감정별 접미사 카운터
emotion_suffix_counter = defaultdict(Counter)

# 5. 세기 2 이상 문장에서 접미사 추출
for _, row in tqdm(df.iterrows(), total=len(df)):
    cleaned = clean_text(row[text_col])
    words = cleaned.split()

    for i in range(0, len(emotion_cols), 2):
        emotion = row[emotion_cols[i]]
        strength = row[emotion_cols[i+1]]

        if pd.notna(emotion) and pd.notna(strength) and int(strength) >= 2:
            for word in words:
                if 3 <= len(word) <= 5:  # 단어 길이 필터
                    suffix = word[-2:]  # 끝 2글자
                    emotion_suffix_counter[emotion].update([suffix])

# 6. 감정별 상위 접미사 Top 10 출력
print("📌 감정별 접미사 Top 10 (3~5글자 단어 대상)")
for emotion, counter in emotion_suffix_counter.items():
    print(f"\n[{emotion}]")
    for suf, count in counter.most_common(10):
        print(f"{suf}: {count}")


100%|██████████| 19374/19374 [00:01<00:00, 11231.62it/s]

📌 감정별 접미사 Top 10 (3~5글자 단어 대상)

[angry]
니까: 438
증나: 436
끼는: 376
으면: 374
라고: 356
에나: 345
이가: 344
한테: 335
테만: 327
는데: 314

[surprise]
랐어: 523
다가: 394
자기: 322
줄이: 244
랬어: 239
는데: 205
피를: 177
라서: 157
피가: 156
었어: 155

[happiness]
는데: 739
났어: 600
디어: 595
었어: 488
하는: 453
라톤: 411
해줘: 404
했어: 404
트에: 365
젝트: 354

[sadness]
울해: 789
는데: 690
니까: 555
문에: 373
지고: 362
로나: 333
이야: 282
어서: 281
하고: 271
겠어: 264

[disgust]
새가: 878
겨운: 565
에서: 279
니까: 220
해서: 219
레가: 213
소를: 200
이가: 197
했어: 180
겠어: 169

[fear]
히게: 371
서워: 331
래도: 153
으로: 150
는데: 128
있어: 117
니까: 105
더니: 104
까지: 99
네가: 99


**3. 감정 라벨별 유사/공통 단어 교집합 분석**

📌 목적: 여러 감정에 공통적으로 나타나는 단어와 특정 감정에만 특화된 단어 분리

🎯 어떻게?

감정별 상위 100 단어 리스트 생성

감정 간 Jaccard 유사도 → 감정 특이 단어와 공통 단어 구분

In [ ]:
import pandas as pd
import re
from collections import Counter, defaultdict
from tqdm import tqdm

# 1. 데이터 불러오기
df = pd.read_csv("/content/5차년도_2차.csv", encoding='cp949')

# 2. 기본 설정
text_col = "발화문"
emotion_cols = ['1번 감정', '1번 감정세기',
                '2번 감정', '2번 감정세기',
                '3번 감정', '3번 감정세기',
                '4번 감정', '4번감정세기',
                '5번 감정', '5번 감정세기']

# 3. 전처리 함수
def clean_text(text):
    return re.sub(r"[^가-힣\s]", "", str(text))

# 4. 감정별 단어 카운터 초기화
emotion_word_counters = defaultdict(Counter)

# 5. 감정세기 ≥ 2 문장에서 단어 수집
for _, row in tqdm(df.iterrows(), total=len(df)):
    cleaned = clean_text(row[text_col])
    words = cleaned.split()

    for i in range(0, len(emotion_cols), 2):
        emotion = row[emotion_cols[i]]
        strength = row[emotion_cols[i+1]]

        if pd.notna(emotion) and pd.notna(strength) and int(strength) >= 2:
            filtered = [w for w in words if len(w) > 1]
            emotion_word_counters[emotion].update(filtered)

# 6. 감정별 Top 100 단어 추출
emotion_top_words = {
    emotion: set([word for word, _ in counter.most_common(100)])
    for emotion, counter in emotion_word_counters.items()
}

# 7. 감정별 고유 단어 계산
emotion_unique_words = {}
all_emotions = set(emotion_top_words.keys())

for emotion in all_emotions:
    others = all_emotions - {emotion}
    other_words = set().union(*(emotion_top_words[o] for o in others))
    unique = emotion_top_words[emotion] - other_words
    emotion_unique_words[emotion] = unique

# 8. 공통 단어 계산
common_words = set.intersection(*emotion_top_words.values())

# 9. 출력 예시: 각 감정별 고유 단어 Top 10 + 공통 단어 15개
print("\n📌 감정별 고유 단어 Top 10")
for emotion, words in emotion_unique_words.items():
    print(f"[{emotion}] {list(words)[:10]}")

print("\n📌 모든 감정 공통 단어 (예시 15개):")
print(list(common_words)[:15])


100%|██████████| 19374/19374 [00:01<00:00, 11516.27it/s]


📌 감정별 고유 단어 Top 10
[surprise] ['멀쩡하던', '쳐서', '달려가서', '일도', '제거제가', '해피', '효과가', '왔기', '끊어지면서', '진정이']
[sadness] ['나가서', '하루종일', '처음', '주식을', '모든', '시작하면서', '샀을', '그런지', '때보다', '집에서']
[angry] ['하니까', '사과', '했다고', '유튜버들이', '유튜버가', '못한', '친절하게', '당해봐서', '새끼', '맨날']
[fear] ['동안에', '몸은', '물에', '주변에', '안에', '밖으로', '홍수', '무서워서', '혼자', '무섭지']
[disgust] ['고약해서', '나왔다', '바로', '시작했어', '상해서', '역겨운', '버려서', '빼놨어', '냄새랑', '사진']
[happiness] ['함께', '발표를', '프로젝트가', '이제', '출전했는데', '생겼어', '뽑는데', '프로젝트', '갱신했어', '좋았어']

📌 모든 감정 공통 단어 (예시 15개):
['거야', '있어', '너무', '내가', '정말', '같아']


**4. 비속어/강조어 단어 감정 연관도 분석**

📌 목적: 욕설이나 강조 단어가 감정 분포에 미치는 영향 → 중요 서브워드 지정

🎯 어떻게?

특정 욕설/강조 표현 포함 여부로 필터링

포함 문장 vs 미포함 문장의 감정 분포 비교 (e.g., 개, 존, 씨발, 졸라 등)

In [ ]:
import pandas as pd
import re
from collections import Counter, defaultdict
from tqdm import tqdm

# 1. 데이터 불러오기
df = pd.read_csv("/content/5차년도_2차.csv", encoding='cp949')

# 2. 욕설/강조어 사전 정의
bad_words = ['씨발', 'ㅅㅂ', 'ㅈ같', '지랄', '개새', '미친', '개', '존나', '졸라', '겁나']

# 3. 감정 컬럼 목록
emotion_cols = ['1번 감정', '1번 감정세기',
                '2번 감정', '2번 감정세기',
                '3번 감정', '3번 감정세기',
                '4번 감정', '4번감정세기',
                '5번 감정', '5번 감정세기']

# 4. 문장에 욕설/강조어가 포함되어 있는지 여부 판단
def has_badword(text):
    return any(bad in text for bad in bad_words)

df['욕설포함'] = df['발화문'].astype(str).apply(has_badword)

# 5. 욕설 포함 여부별 감정 세기 평균 계산
emotion_strength_data = defaultdict(lambda: {'욕설': [], '비욕설': []})

for _, row in tqdm(df.iterrows(), total=len(df)):
    is_bad = '욕설' if row['욕설포함'] else '비욕설'

    for i in range(0, len(emotion_cols), 2):
        emotion = row[emotion_cols[i]]
        strength = row[emotion_cols[i+1]]

        if pd.notna(emotion) and pd.notna(strength):
            emotion_strength_data[emotion][is_bad].append(int(strength))

# 6. 결과 요약 출력
print("\n📊 감정별 욕설 포함/비포함 평균 감정 세기")
for emotion, data in emotion_strength_data.items():
    avg_bad = sum(data['욕설']) / len(data['욕설']) if data['욕설'] else 0
    avg_clean = sum(data['비욕설']) / len(data['비욕설']) if data['비욕설'] else 0
    print(f"{emotion:<10} | 욕설 포함: {avg_bad:.2f} | 비포함: {avg_clean:.2f}")


100%|██████████| 19374/19374 [00:01<00:00, 13018.51it/s]


📊 감정별 욕설 포함/비포함 평균 감정 세기
angry      | 욕설 포함: 1.71 | 비포함: 1.39
surprise   | 욕설 포함: 1.65 | 비포함: 1.42
happiness  | 욕설 포함: 1.08 | 비포함: 1.29
neutral    | 욕설 포함: 0.00 | 비포함: 0.00
sadness    | 욕설 포함: 1.16 | 비포함: 1.30
disgust    | 욕설 포함: 1.51 | 비포함: 1.35
fear       | 욕설 포함: 1.30 | 비포함: 1.33


**5. 자주 등장하는 감정 서술 어미/조사/표현 분석**

📌 목적: -네, -야, -잖아, -구나 같은 감정 표현 스타일을 분석 → 서브워드화 가능성 탐색

🎯 어떻게?

문장 끝 2~4글자 n-gram 추출 후 빈도/감정별 매핑 분석

In [ ]:
import pandas as pd
import re
from collections import Counter, defaultdict
from tqdm import tqdm

# 1. 데이터 불러오기
df = pd.read_csv("/content/5차년도_2차.csv", encoding='cp949')

# 2. 감정 컬럼
text_col = "발화문"
emotion_cols = ['1번 감정', '1번 감정세기',
                '2번 감정', '2번 감정세기',
                '3번 감정', '3번 감정세기',
                '4번 감정', '4번감정세기',
                '5번 감정', '5번 감정세기']

# 3. 전처리 함수
def clean_text(text):
    return re.sub(r"[^가-힣\s]", "", str(text)).strip()

# 4. 감정별 문장 끝부분 n-gram 추출
emotion_suffix_ngrams = defaultdict(Counter)

for _, row in tqdm(df.iterrows(), total=len(df)):
    text = clean_text(row[text_col])
    if len(text) < 2:
        continue

    # 문장 끝부분에서 2~4글자 추출
    end = text[-4:]
    suffixes = [end[-n:] for n in range(2, min(len(end), 4) + 1)]

    for i in range(0, len(emotion_cols), 2):
        emotion = row[emotion_cols[i]]
        strength = row[emotion_cols[i+1]]

        if pd.notna(emotion) and pd.notna(strength) and int(strength) >= 2:
            emotion_suffix_ngrams[emotion].update(suffixes)

# 5. 감정별 자주 쓰이는 문장 끝 n-gram 출력
print("\n📌 감정별 자주 쓰이는 문장 어미 (2~4글자)")
for emotion, counter in emotion_suffix_ngrams.items():
    print(f"\n[{emotion}]")
    for suf, count in counter.most_common(10):
        print(f"{suf}: {count}")



100%|██████████| 19374/19374 [00:01<00:00, 11589.78it/s]


📌 감정별 자주 쓰이는 문장 어미 (2~4글자)

[angry]
이야: 262
않아: 215
 않아: 215
왔어: 181
그래: 166
 그래: 165
거야: 154
겠어: 137
없어: 129
 없어: 129

[surprise]
랐어: 516
놀랐어: 513
 놀랐어: 510
랬어: 231
놀랬어: 229
 놀랬어: 225
됐어: 175
거야: 151
첨됐어: 124
당첨됐어: 124

[happiness]
났어: 514
끝났어: 514
 끝났어: 514
었어: 447
됐어: 355
했어: 323
되었어: 301
있어: 295
 있어: 292
거야: 248

[sadness]
울해: 755
우울해: 752
 우울해: 752
있어: 254
없어: 241
 없어: 241
이야: 235
같아: 230
 같아: 228
겠어: 225

[disgust]
있어: 398
 있어: 373
고 있어: 360
했어: 185
왔어: 182
작했어: 150
시작했어: 150
나왔어: 146
않아: 143
 않아: 143

[fear]
같아: 401
 같아: 401
것 같아: 310
서워: 235
무서워: 235
 무서워: 235
있어: 104
없어: 94
 없어: 94
거 같아: 91


✨ 이 분석을 통해 BPE 방식이 적합하다고 판단한 이유:
WordPiece는 자주 등장하는 "전체 단어 단위" 우선 → 감정 표현 파생형 처리에 불리

BPE는 "부분 단위 빈도 기반 병합" → 감정 접두사(개-), 접미사(-했어), 강조어(완전-), 감정 강조 이모티콘(ㅎㅎ, ㅠㅠ 등) 처리에 유리

우리 데이터셋 특성상 띄어쓰기 오류, 신조어, 강조 표현이 많음 → BPE가 더 잘 대응함

| 항목          | WordPiece | BPE |
| ----------- | --------- | --- |
| 전체 단어 기반    | ✅         | ❌   |
| 접두사/접미사 분해  | ❌         | ✅   |
| 신조어 대응      | ❌         | ✅   |
| 띄어쓰기 오류 대응  | ❌         | ✅   |
| 감정 강조 표현 반영 | ❌         | ✅   |
